# Week 6: Abstract Classes & Interfaces — PHASE 2: Composing Components

*Object Oriented Programming . 3 Hours . Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

| # | Objective |
|---|---|
| 1 | Understand the problem of **missing methods** in subclasses |
| 2 | Use the `abc` module to define **Abstract Base Classes** |
| 3 | Apply the `@abstractmethod` decorator to enforce method contracts |
| 4 | Implement abstract classes in concrete subclasses |
| 5 | Understand why **interfaces** matter in real engineering systems |
| 6 | Design your own **interface** for a data processing pipeline |
| 7 | Avoid **common mistakes** when working with abstract classes |

## 🎯 Core Mastery Connection

Interfaces are contracts that guarantee a component provides certain methods. This makes composition safe — when you plug components together, you know each one will behave as promised. This week you learn to define those contracts using abstract classes, so that any component meeting the contract can be safely composed into a larger system.

---

## Part 1: The Problem — Guaranteeing Methods Exist

Last week, we learned that child classes can **override** parent methods. But what if someone **forgets** to override an important method?

Let's see the problem:

**Figure 6.1** — Sensor class example

In [ ]:
# The problem: nothing forces a child to implement display()
class Sensor:
    def __init__(self, name):
        self.name = name
        self.value = 0.0

    def display(self):
        """Generic display — not very useful."""
        print(f"{self.name}: {self.value}")

class VibrationSensor(Sensor):
    def __init__(self, name):
        super().__init__(name)
    # Oops! Forgot to override display()
    # No error — it just uses the parent's generic version

vs = VibrationSensor("Accel-X")
vs.value = 2.5
vs.display()  # works, but uses the wrong display format!

In [ ]:
# This is a real problem in bigger projects:
# - A team agrees that every sensor MUST have its own display()
# - Someone creates a new sensor and forgets
# - The code runs without error, but the output is wrong
# - The bug is hard to find!

# We need a way to say:
# "You MUST implement display() — or the code won't run at all."
print("We need a way to enforce method implementation!")

---

## Part 2: Abstract Base Classes (ABC)

Python provides the `abc` module to solve this problem.

An **Abstract Base Class (ABC)** is a class that:
- Cannot be created directly (no instances)
- Defines methods that **must** be implemented by all children
- Acts as a **contract** or **blueprint**

**Figure 2.1** — Abstract vs Concrete Classes

```
  Sensor (ABSTRACT)          ← cannot create instances
     │
     ├── TemperatureSensor   ← concrete: must implement all abstract methods
     └── PressureSensor      ← concrete: must implement all abstract methods
```

| Term | Meaning |
|---|---|
| **Abstract class** | A class with at least one abstract method — cannot be instantiated |
| **Concrete class** | A regular class that CAN be instantiated |
| **Abstract method** | A method declared but not implemented — children must implement it |

In [ ]:
# Import the tools we need
from abc import ABC, abstractmethod

# ABC  → the base class that makes a class abstract
# abstractmethod → the decorator that marks a method as "must implement"

print("abc module imported successfully!")
print(f"ABC is: {ABC}")

---

## Part 3: The `@abstractmethod` Decorator

To create an abstract class:
1. Inherit from `ABC`
2. Mark methods with `@abstractmethod`

Any class with abstract methods **cannot be instantiated** until ALL abstract methods are implemented.

**Figure 6.2** — Abstract base class with @abstractmethod

In [ ]:
from abc import ABC, abstractmethod

class Sensor(ABC):  # inherit from ABC to make it abstract
    def __init__(self, name):
        self.name = name
        self.value = 0.0

    @abstractmethod
    def display(self):
        """Every sensor MUST implement its own display."""
        pass

    @abstractmethod
    def status(self):
        """Every sensor MUST report its status."""
        pass

    def read(self):
        """This is a normal method — children inherit it as-is."""
        return self.value

# Try to create an instance of the abstract class
try:
    s = Sensor("test")  # this will FAIL
except TypeError as e:
    print(f"Error: {e}")
    print("→ You cannot create an instance of an abstract class!")

**Figure 6.3** — IncompleteTemperatureSensor class example

In [ ]:
# What happens if a child only implements SOME abstract methods?
class IncompleteTemperatureSensor(Sensor):
    def __init__(self, name):
        super().__init__(name)

    def display(self):  # implemented display...
        print(f"Temp: {self.value}°C")
    # But forgot status()!

try:
    ts = IncompleteTemperatureSensor("TMP36")
except TypeError as e:
    print(f"Error: {e}")
    print("→ Must implement ALL abstract methods!")

---

## Part 4: Implementing Abstract Classes

A child class must implement **every** abstract method to become a concrete (usable) class.

**Figure 4.1** — The contract pattern

```
Sensor (ABC)
├── display()    @abstractmethod  ← contract: must implement
├── status()     @abstractmethod  ← contract: must implement
└── read()       (normal method)  ← inherited as-is
```

In [ ]:
from abc import ABC, abstractmethod

class Sensor(ABC):
    def __init__(self, name, unit):
        self.name = name
        self.unit = unit
        self.value = 0.0

    @abstractmethod
    def display(self):
        """Show the sensor reading in a sensor-specific format."""
        pass

    @abstractmethod
    def status(self):
        """Return a status string: 'OK', 'WARNING', or 'ERROR'."""
        pass

    def read(self):
        """Return the current value (inherited by all children)."""
        return self.value

**Figure 6.4** — TemperatureSensor class example

In [ ]:
class TemperatureSensor(Sensor):
    def __init__(self, name, max_temp=100):
        super().__init__(name, "°C")  # call parent init
        self.max_temp = max_temp

    def display(self):  # ✅ implements abstract method
        print(f"🌡️ {self.name}: {self.value} {self.unit}")

    def status(self):  # ✅ implements abstract method
        if self.value > self.max_temp:
            return "ERROR"
        elif self.value > self.max_temp * 0.8:
            return "WARNING"
        return "OK"

# Now we CAN create an instance
ts = TemperatureSensor("Engine-Temp", max_temp=120)
ts.value = 95
ts.display()
print(f"Status: {ts.status()}")
print(f"Reading: {ts.read()}")  # inherited from Sensor

**Figure 6.5** — PressureSensor class example

In [ ]:
class PressureSensor(Sensor):
    def __init__(self, name, max_pressure=1000):
        super().__init__(name, "kPa")
        self.max_pressure = max_pressure

    def display(self):  # ✅ implements abstract method
        pct = (self.value / self.max_pressure) * 100
        print(f"📊 {self.name}: {self.value} {self.unit} ({pct:.0f}%)")

    def status(self):  # ✅ implements abstract method
        if self.value > self.max_pressure:
            return "ERROR"
        elif self.value > self.max_pressure * 0.9:
            return "WARNING"
        return "OK"

ps = PressureSensor("Hydraulic", max_pressure=500)
ps.value = 480
ps.display()
print(f"Status: {ps.status()}")

---

## Part 5: Why Interfaces Matter

In many languages (Java, C#), there is a formal concept called an **interface**. Python uses **abstract classes** to achieve the same goal.

An interface is like a **contract** that says:

> *"Any class that implements this interface promises to have these methods."*

### Real-world analogy

Think of a **USB port**:
- The USB standard defines the interface (shape, pins, protocols)
- Any device that follows the standard can plug in
- You don't need to know if it's a mouse, keyboard, or flash drive

**Figure 5.1** — Interface as a contract

```
Interface: DataProcessor
├── process(data)     ← every implementation must have this
└── get_result()      ← every implementation must have this

Implementations:
├── AverageProcessor  ← processes data by averaging
├── MaxProcessor      ← processes data by finding max
└── FilterProcessor   ← processes data by filtering
```

| Benefit | Explanation |
|---|---|
| **Consistency** | Every implementation has the same methods |
| **Swappability** | You can swap one implementation for another |
| **Error prevention** | Missing methods are caught immediately |
| **Team collaboration** | Everyone agrees on the interface first |

In [ ]:
from abc import ABC, abstractmethod

# Define an interface (abstract class with ONLY abstract methods)
class DataProcessor(ABC):
    """Interface: any data processor must implement these methods."""

    @abstractmethod
    def process(self, data):
        """Process a list of numbers."""
        pass

    @abstractmethod
    def get_result(self):
        """Return the processing result."""
        pass

print("DataProcessor interface defined.")
print("Any class that inherits from it MUST implement process() and get_result().")

**Figure 6.6** — AverageProcessor class example

In [ ]:
# Implementation 1: calculates the average
class AverageProcessor(DataProcessor):
    def __init__(self):
        self.result = 0.0

    def process(self, data):  # ✅
        self.result = sum(data) / len(data)

    def get_result(self):  # ✅
        return self.result

# Implementation 2: finds the maximum
class MaxProcessor(DataProcessor):
    def __init__(self):
        self.result = 0.0

    def process(self, data):  # ✅
        self.result = max(data)

    def get_result(self):  # ✅
        return self.result

# Both follow the same interface!
data = [10, 25, 30, 15, 20]

for processor in [AverageProcessor(), MaxProcessor()]:
    processor.process(data)
    print(f"{type(processor).__name__}: {processor.get_result()}")

---

## Part 6: Designing a Filter Interface

In signal processing and mechatronics, **filters** are essential. Let's design a `Filter` interface that any filter implementation must follow.

**Figure 6.1** — Filter interface design

```
Filter (ABC)
├── apply(value)     → process one value, return filtered result
├── reset()          → clear internal state
└── get_name()       → return the filter's name
```

In [ ]:
from abc import ABC, abstractmethod

class Filter(ABC):
    """Interface for signal filters.

    Any filter must be able to:
    - Apply itself to a single value
    - Reset its internal state
    - Report its name
    """

    @abstractmethod
    def apply(self, value):
        """Process one value and return the filtered result."""
        pass

    @abstractmethod
    def reset(self):
        """Clear any internal state (e.g., history buffer)."""
        pass

    @abstractmethod
    def get_name(self):
        """Return the name of this filter."""
        pass

print("Filter interface defined with 3 abstract methods.")

---

## Part 7: Multiple Implementations

Now let's create several concrete filter classes that implement the `Filter` interface.

**Figure 6.7** — Filter interface implementation

In [ ]:
class MovingAverageFilter(Filter):
    """Smooths data by averaging the last N values."""

    def __init__(self, window_size=3):
        self.window_size = window_size  # how many values to average
        self.buffer = []                # stores recent values

    def apply(self, value):  # ✅ implements abstract method
        self.buffer.append(value)       # add new value
        # Keep only the last window_size values
        if len(self.buffer) > self.window_size:
            self.buffer.pop(0)          # remove oldest
        # Return the average of the buffer
        return sum(self.buffer) / len(self.buffer)

    def reset(self):  # ✅ implements abstract method
        self.buffer = []

    def get_name(self):  # ✅ implements abstract method
        return f"MovingAverage(window={self.window_size})"

# Test it
ma = MovingAverageFilter(window_size=3)
print(f"Filter: {ma.get_name()}")
for val in [10, 20, 30, 40, 50]:
    result = ma.apply(val)
    print(f"  Input: {val} → Output: {result:.1f}")

**Figure 6.8** — Filter interface implementation

In [ ]:
class ThresholdFilter(Filter):
    """Clamps values to stay within a min/max range."""

    def __init__(self, min_val=0, max_val=100):
        self.min_val = min_val  # minimum allowed value
        self.max_val = max_val  # maximum allowed value

    def apply(self, value):  # ✅
        # Clamp the value between min and max
        if value < self.min_val:
            return self.min_val
        elif value > self.max_val:
            return self.max_val
        return value

    def reset(self):  # ✅
        pass  # no state to reset

    def get_name(self):  # ✅
        return f"Threshold({self.min_val}-{self.max_val})"

# Test it
tf = ThresholdFilter(min_val=0, max_val=50)
print(f"Filter: {tf.get_name()}")
for val in [-10, 25, 75, 100]:
    result = tf.apply(val)
    print(f"  Input: {val} → Output: {result}")

**Figure 6.9** — Filter interface implementation

In [ ]:
class ScalingFilter(Filter):
    """Multiplies values by a scale factor."""

    def __init__(self, factor=1.0):
        self.factor = factor  # multiplication factor

    def apply(self, value):  # ✅
        return value * self.factor

    def reset(self):  # ✅
        pass  # no state to reset

    def get_name(self):  # ✅
        return f"Scaling(x{self.factor})"

# Test it
sf = ScalingFilter(factor=2.5)
print(f"Filter: {sf.get_name()}")
for val in [10, 20, 30]:
    result = sf.apply(val)
    print(f"  Input: {val} → Output: {result}")

**Figure 6.10** — Function example

In [ ]:
# Polymorphism: use ANY filter the same way
def process_signal(raw_data, filter_obj):
    """Apply a filter to raw signal data.

    Works with ANY filter that implements the Filter interface.
    """
    filter_obj.reset()  # start fresh
    results = []
    for value in raw_data:
        results.append(filter_obj.apply(value))
    return results

# Simulated noisy sensor data
raw = [10, 12, 50, 11, 13, 80, 12, 14, 11]

# Try different filters — same function, different behavior
filters = [
    MovingAverageFilter(window_size=3),
    ThresholdFilter(min_val=0, max_val=30),
    ScalingFilter(factor=0.1),
]

print(f"Raw data: {raw}\n")
for f in filters:
    output = process_signal(raw, f)
    rounded = [round(x, 1) for x in output]
    print(f"{f.get_name():30s} → {rounded}")

---

## Part 8: Common Mistakes

| Mistake | What happens | Fix |
|---|---|---|
| Forget to inherit from `ABC` | No enforcement — abstract methods are just regular methods | Add `(ABC)` to the class |
| Forget `@abstractmethod` | The method is not enforced — children can skip it | Add the decorator |
| Try to instantiate an ABC | `TypeError` at runtime | Only instantiate concrete children |
| Implement only some abstract methods | `TypeError` at runtime | Implement ALL abstract methods |
| Misspell the method name | Not recognized as implementing the abstract method | Use exact same name |

**Figure 6.11** — Abstract base class with @abstractmethod

In [ ]:
# Mistake 1: forgetting ABC
class BadInterface:  # NOT abstract — missing (ABC)
    @abstractmethod
    def do_something(self):
        pass

# This SHOULD fail, but it doesn't — because it's not a real ABC
b = BadInterface()
print("Oops! BadInterface was created even though do_something is 'abstract'.")
print("Fix: class BadInterface(ABC):")

**Figure 6.12** — Abstract base class with @abstractmethod

In [ ]:
# Mistake 2: misspelling the method name
class Actuator(ABC):
    @abstractmethod
    def activate(self):
        pass

class Valve(Actuator):
    def activte(self):  # typo! 'activte' instead of 'activate'
        print("Valve opened")

try:
    v = Valve()
except TypeError as e:
    print(f"Error: {e}")
    print("→ 'activte' does not match 'activate' — the contract is not fulfilled!")

**Figure 6.13** — Valve class example

---

## Studio: Defining a Filter Interface

> **Composition in action:** The `FilterPipeline` below composes multiple `Filter` components into a chain. Each filter is a self-contained component that follows the `Filter` interface contract. Because every filter guarantees `apply()`, `reset()`, and `get_name()`, the pipeline can safely plug them together in any order.

Let's build a signal processing pipeline using our `Filter` interface.

---

## 🛠️ Studio: Defining a Filter Interface

Let's build a signal processing pipeline using our `Filter` interface.

**Figure 6.14** — Abstract base class with @abstractmethod

In [ ]:
from abc import ABC, abstractmethod

# The Filter interface (same as Part 6)
class Filter(ABC):
    @abstractmethod
    def apply(self, value):
        pass

    @abstractmethod
    def reset(self):
        pass

    @abstractmethod
    def get_name(self):
        pass


class FilterPipeline:
    """Chains multiple filters together.

    Data flows through each filter in order.
    Works with ANY class that implements the Filter interface.
    """
    def __init__(self):
        self.filters = []  # list of Filter objects

    def add_filter(self, f):
        """Add a filter to the pipeline."""
        if not isinstance(f, Filter):
            raise TypeError("Only Filter objects can be added!")
        self.filters.append(f)

    def process(self, raw_data):
        """Run data through all filters in sequence."""
        # Reset all filters
        for f in self.filters:
            f.reset()

        results = []
        for value in raw_data:
            current = value
            # Pass through each filter in order
            for f in self.filters:
                current = f.apply(current)
            results.append(current)
        return results

    def describe(self):
        """Show the pipeline configuration."""
        names = [f.get_name() for f in self.filters]
        print("Pipeline: " + " → ".join(names))

**Figure 6.15** — Filter interface implementation

In [ ]:
# Redefine our filter implementations
class MovingAverageFilter(Filter):
    def __init__(self, window_size=3):
        self.window_size = window_size
        self.buffer = []

    def apply(self, value):
        self.buffer.append(value)
        if len(self.buffer) > self.window_size:
            self.buffer.pop(0)
        return sum(self.buffer) / len(self.buffer)

    def reset(self):
        self.buffer = []

    def get_name(self):
        return f"MA({self.window_size})"


class ThresholdFilter(Filter):
    def __init__(self, min_val=0, max_val=100):
        self.min_val = min_val
        self.max_val = max_val

    def apply(self, value):
        return max(self.min_val, min(self.max_val, value))

    def reset(self):
        pass

    def get_name(self):
        return f"Clamp({self.min_val}-{self.max_val})"


class ScalingFilter(Filter):
    def __init__(self, factor=1.0):
        self.factor = factor

    def apply(self, value):
        return value * self.factor

    def reset(self):
        pass

    def get_name(self):
        return f"Scale(x{self.factor})"

---

## Exercises

> **Composition lens:** In these exercises, you are building components with guaranteed interfaces. Every abstract class you define is a contract that makes future composition safe and predictable.

### Difficulty Guide

| Level | Meaning |
|---|---|
| Easy | Apply what you learned directly |
| Medium | Combine concepts or add small twists |
| Challenge | Think deeper, design your own solution |

### 🟢 Exercise 1 — Your First ABC

Create an abstract class `Shape` with:
- An abstract method `area()` that returns the area
- An abstract method `describe()` that prints a description

Then create a `Rectangle` class that implements both methods.

<details><summary>💡 Hint</summary>

```python
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass
```

</details>

In [ ]:
# ✏️ [EX1] Your First ABC



### 🟢 Exercise 2 — Cannot Instantiate

Try to create an instance of `Shape` directly. Catch the `TypeError` and print the error message.
This shows that abstract classes cannot be instantiated.

<details><summary>💡 Hint</summary>

Use a try/except block around `Shape()`.

</details>

In [ ]:
# ✏️ [EX2] Cannot Instantiate



### 🟢 Exercise 3 — Multiple Implementations

Using your `Shape` ABC from EX1, create two more implementations:
- `Circle` with a `radius` parameter
- `Triangle` with `base` and `height` parameters

Create one of each and print their areas.

<details><summary>💡 Hint</summary>

Circle area = 3.14159 * radius ** 2. Triangle area = 0.5 * base * height.

</details>

In [ ]:
# ✏️ [EX3] Multiple Implementations



### 🟡 Exercise 4 — Actuator Interface

Create an abstract class `Actuator` with:
- `activate()` — abstract method
- `deactivate()` — abstract method
- `is_active()` — abstract method that returns a boolean

Create two concrete classes: `Valve` and `Relay`, each implementing all three methods.

<details><summary>💡 Hint</summary>

Use a `self.active` boolean attribute to track state. Set it to `True` in `activate()` and `False` in `deactivate()`.

</details>

In [ ]:
# ✏️ [EX4] Actuator Interface



### 🟡 Exercise 5 — Mixed Abstract and Concrete Methods

Create an abstract class `Logger` with:
- An abstract method `log(message)` — how the message is logged depends on the child
- A concrete method `log_error(message)` — calls `self.log("ERROR: " + message)`
- A concrete method `log_info(message)` — calls `self.log("INFO: " + message)`

Create a `ConsoleLogger` that prints the message, and a `FileLogger` that appends messages to a list.

<details><summary>💡 Hint</summary>

The concrete methods `log_error` and `log_info` are inherited by children. They call `self.log()`, which will use the child's implementation.

</details>

In [ ]:
# ✏️ [EX5] Mixed Abstract and Concrete Methods



### 🟡 Exercise 6 — Incomplete Implementation

Using the `Filter` interface from the lecture, create a class `BrokenFilter` that only implements `apply()` and `get_name()` but NOT `reset()`.

Try to create an instance and observe the error.

<details><summary>💡 Hint</summary>

Use try/except to catch the `TypeError` and print what method is missing.

</details>

In [ ]:
# ✏️ [EX6] Incomplete Implementation



### 🟡 Exercise 7 — Custom Filter

Create a `DeadbandFilter` that implements the `Filter` interface.

A deadband filter ignores small changes: it only updates the output when the input changes by more than a `threshold` from the last output.

Example with threshold=5:
- Input: 10 → Output: 10 (first value)
- Input: 12 → Output: 10 (change of 2, below threshold)
- Input: 16 → Output: 16 (change of 6, above threshold)

<details><summary>💡 Hint</summary>

Store `self.last_output`. In `apply()`, check if `abs(value - self.last_output) > self.threshold`. If yes, update and return the new value. If no, return the old output.

</details>

In [ ]:
# ✏️ [EX7] Custom Filter



### 🟡 Exercise 8 — isinstance with ABCs

Create a list containing objects of different types: `MovingAverageFilter`, `ThresholdFilter`, `ScalingFilter`, and a plain string `"not a filter"`.

Write a function `count_filters(items)` that counts how many items in the list are instances of `Filter`.

<details><summary>💡 Hint</summary>

Use `isinstance(item, Filter)` — this works with the abstract base class and catches all subclasses.

</details>

In [ ]:
# ✏️ [EX8] isinstance with ABCs



### 🔴 Exercise 9 — Controller Interface

Design an abstract class `Controller` with:
- `set_target(value)` — abstract
- `compute(current_value)` — abstract, returns the control signal
- `reset()` — abstract

Create a `BangBangController` (on/off controller):
- If `current_value < target`: return 1.0 (full on)
- If `current_value >= target`: return 0.0 (off)

Test it by simulating a heater that tries to reach 50°C.

<details><summary>💡 Hint</summary>

```python
temp = 20.0
controller.set_target(50)
for step in range(20):
    signal = controller.compute(temp)
    temp += signal * 3 - 1  # heats if on, cools slightly if off
```

</details>

In [ ]:
# ✏️ [EX9] Controller Interface



### 🔴 Exercise 10 — Pipeline Builder

Extend the `FilterPipeline` from the studio with a `remove_filter(name)` method that removes a filter by its name (using `get_name()`).

Also add a `__len__()` method that returns the number of filters.

Test by adding 3 filters, removing one by name, and checking the length.

<details><summary>💡 Hint</summary>

Loop through `self.filters`, find the one where `f.get_name() == name`, and remove it with `self.filters.remove(f)`.

</details>

In [ ]:
# ✏️ [EX10] Pipeline Builder



### 🔴 Exercise 11 — Sensor + Filter Integration

Create an abstract `SmartSensor` class that combines a sensor and a filter:
- Has `name` and `filter` (a Filter object) attributes
- Abstract method: `raw_read()` — returns a raw reading
- Concrete method: `filtered_read()` — calls `raw_read()` and passes it through `self.filter.apply()`

Create a `NoisyTemperatureSensor` that implements `raw_read()` by returning a base temperature plus some random noise.

<details><summary>💡 Hint</summary>

```python
import random

def raw_read(self):
    return self.base_temp + random.uniform(-5, 5)
```

</details>

In [ ]:
# ✏️ [EX11] Sensor + Filter Integration



---

## 🌉 Bridge to Next Week

This week we learned how to define **contracts** using abstract classes and how to build **swappable components** that all follow the same interface.

These are foundational ideas in software engineering:
- **Abstract classes** define what must exist
- **Concrete classes** define how it works
- **Interfaces** let us write code that works with any implementation

**Next week**, we will explore more OOP patterns and learn how to handle **errors and exceptions** properly in object-oriented programs.

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_06"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")